# 03 — Analyse des résultats par nuance politique

Distribution des voix par nuance politique, classements par département,
et carte d'implantation des partis.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_FILE = Path('../../public/cities/cities-data.json')

with open(DATA_FILE) as f:
    raw = json.load(f)

cities = list(raw.values())
print(f"{len(cities)} communes")

In [ ]:
# Aplatir tous les résultats Tour 1 en lignes
result_rows = []
for c in cities:
    t1 = c.get('Tour 1', {})
    exprimes = t1.get('Exprimés', 0) or 1
    inscrits = t1.get('Inscrits', 0) or 1
    for r in t1.get('resultats', []):
        result_rows.append({
            'commune': c['nom_standard'],
            'code_dept': c['code_departement'],
            'dept': c['libelle_departement'],
            'nuance': r.get('Code Nuance', '?'),
            'liste': r.get('Liste', ''),
            'voix': r.get('Voix', 0),
            'pct_exprimes': r.get('% Voix/Exp', 0),
            'pct_inscrits': r.get('% Voix/Ins', 0),
            'sieges': r.get('Sièges / Elu', 0),
            'inscrits': inscrits,
        })

df = pd.DataFrame(result_rows)
print(f"{len(df):,} lignes de résultats (listes × communes)")
print(f"\n{df['nuance'].nunique()} nuances politiques distinctes")
print(df['nuance'].value_counts().head(20))

## Voix totales par nuance politique

In [ ]:
nuances_total = (
    df.groupby('nuance')['voix']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
nuances_total['pct'] = nuances_total['voix'] / nuances_total['voix'].sum() * 100

fig, ax = plt.subplots(figsize=(13, 6))
colors = plt.cm.tab20(np.linspace(0, 1, len(nuances_total)))
bars = ax.bar(nuances_total['nuance'], nuances_total['voix'] / 1e6, color=colors, edgecolor='white')
ax.set_xlabel("Nuance politique")
ax.set_ylabel("Voix (millions)")
ax.set_title("Total des voix par nuance politique — Tour 1")
ax.tick_params(axis='x', rotation=45)

for bar, pct in zip(bars, nuances_total['pct']):
    if pct > 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/03_voix_nuances.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 nuances par voix :")
print(nuances_total.head(10)[['nuance', 'voix', 'pct']].to_string(index=False))

## Nuance dominante par département

In [ ]:
dept_nuance = (
    df.groupby(['code_dept', 'dept', 'nuance'])['voix']
    .sum()
    .reset_index()
)
dept_dominant = dept_nuance.loc[dept_nuance.groupby('code_dept')['voix'].idxmax()]

print(f"Répartition des nuances dominantes par département :")
print(dept_dominant['nuance'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(8, 5))
dept_dominant['nuance'].value_counts().plot(kind='bar', ax=ax, color=plt.cm.Set2(np.linspace(0, 1, dept_dominant['nuance'].nunique())), edgecolor='white')
ax.set_title("Nuance politique dominante par département")
ax.set_xlabel("Nuance")
ax.set_ylabel("Nombre de départements")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/03_nuances_dominantes_dept.png', dpi=150, bbox_inches='tight')
plt.show()

## Score moyen des nuances selon la taille des communes

In [ ]:
# Focus sur les 8 nuances les plus représentées
top_nuances = nuances_total.head(8)['nuance'].tolist()
df_top = df[df['nuance'].isin(top_nuances)].copy()
df_top['size_cat'] = pd.cut(
    df_top['inscrits'],
    bins=[0, 500, 2000, 10000, 50000, float('inf')],
    labels=['< 500', '500–2k', '2k–10k', '10k–50k', '> 50k']
)

pivot = df_top.groupby(['size_cat', 'nuance'])['pct_exprimes'].mean().unstack('nuance').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.8)
ax.set_title("Score moyen (% exprimés) par nuance selon la taille de la commune")
ax.set_xlabel("Taille de la commune (inscrits)")
ax.set_ylabel("% des exprimés (moyenne)")
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Nuance', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/03_scores_par_taille.png', dpi=150, bbox_inches='tight')
plt.show()